# 03 - Carga no Azure SQL Server
**Squad 1 — Data Quality em Tempo Real | Dupla 1**
**Integrantes:** Gabriel Franz Simoni & Carlos Eduardo Santos de Souza
**Tabelas destino:** `squad1.ecommerce_itens_pedido` e `squad1.ecommerce_rastreamento_entregas`

### Regras/KPIs gerados por este notebook:
Nenhum. Este notebook não aplica regras de qualidade — grava os DataFrames já consolidados/analisados no `02_analise_exploratoria` no Azure SQL Server, e valida a gravação comparando a contagem de linhas.

### Objetivo (Task 4):
Gravar os DataFrames consolidados e analisados (`02_analise_exploratoria.ipynb`) no schema `squad1` do Azure SQL Server, e validar a gravação lendo de volta.

**Pré-requisito:** `02_analise_exploratoria.ipynb` já executado nessa sessão (precisamos de `df_itens_pedido_rt` e `df_rastreamento_rt` em memória — como cada notebook é uma sessão separada, é melhor rodar tudo de novo aqui, do zero, pra esse notebook funcionar sozinho).


In [0]:
%run /Workspace/Users/gabrielfranzsimoni@gmail.com/estagio-empregadados-turma-2/squad1/02_analise_exploratoria_documentado


### Etapa 1 — Credenciais do SQL Server

Pega as credenciais do mesmo Secret Scope (`internship-squad1`), chaves específicas do SQL Server.


In [0]:
jdbc_host = dbutils.secrets.get(scope="internship-squad1", key="sql-host")
jdbc_database = dbutils.secrets.get(scope="internship-squad1", key="sql-database")
jdbc_username = dbutils.secrets.get(scope="internship-squad1", key="sql-username")
jdbc_password = dbutils.secrets.get(scope="internship-squad1", key="sql-password")


### Etapa 2 — Gravação (overwrite) no SQL Server

Grava as duas tabelas via conector nativo `format("sqlserver")`, modo `overwrite` (recria a tabela a cada execução).


In [0]:
(df_itens_pedido_rt.write
    .format("sqlserver")
    .mode("overwrite")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("dbtable", "squad1.ecommerce_itens_pedido")
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .option("encrypt", "true")
    .save())
print("squad1.ecommerce_itens_pedido gravada.")

(df_rastreamento_rt.write
    .format("sqlserver")
    .mode("overwrite")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("dbtable", "squad1.ecommerce_rastreamento_entregas")
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .option("encrypt", "true")
    .save())
print("squad1.ecommerce_rastreamento_entregas gravada.")


### Etapa 3 — Validação

Lê as mesmas tabelas de volta do SQL Server e compara a contagem de linhas com o que foi gravado, pra confirmar que não houve perda de dado na carga.


In [0]:
df_verifica_itens = (spark.read
    .format("sqlserver")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("dbtable", "squad1.ecommerce_itens_pedido")
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .option("encrypt", "true")
    .load())

df_verifica_rastreamento = (spark.read
    .format("sqlserver")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("dbtable", "squad1.ecommerce_rastreamento_entregas")
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .option("encrypt", "true")
    .load())

print(f"squad1.ecommerce_itens_pedido no SQL Server: {df_verifica_itens.count()} linhas (esperado: {df_itens_pedido_rt.count()})")
print(f"squad1.ecommerce_rastreamento_entregas no SQL Server: {df_verifica_rastreamento.count()} linhas (esperado: {df_rastreamento_rt.count()})")
